In [1]:
import h5py 
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, accuracy_score

# ========== 1. SPLIT DATA ==========
def split_data(X, y, test_size=0.2, random_state=42):
    return train_test_split(X, y, test_size=test_size, stratify=y, random_state=random_state)

# ========== 2. HYPERPARAMETER TUNING ==========
def cross_validate_svm(X_train, y_train, cv=5):
    """
    Perform cross-validation for SVM hyperparameters (C and kernel).
    """
    param_grid = {
        'estimator__C': [0.1, 1, 10],
    }
    
    # dual=True when n_samples < n_features 
    ovr = OneVsRestClassifier(svm.LinearSVC(max_iter=1000, dual=True, # dual=False,
                                            random_state=0))

    grid_search = GridSearchCV(ovr, param_grid, cv=cv, n_jobs=-1, scoring='accuracy')
    grid_search.fit(X_train, y_train)

    # Cross-validate best model to compute average accuracy
    best_model = grid_search.best_estimator_
    cv_scores = cross_val_score(best_model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)

    mean_accuracy = np.mean(cv_scores)
    std_accuracy = np.std(cv_scores)
    mean_error = 1.0 - mean_accuracy

    return best_model, grid_search.best_params_, mean_accuracy, mean_error, std_accuracy

# ========== 3. MAIN FUNCTION ==========
def run_one_vs_rest_svm(X, y):
    # Step 1: Train-test split
    X_train, X_test, y_train, y_test = split_data(X, y)

    # Step 2: Cross-validation
    print("Running cross-validation and hyperparameter tuning...")
    best_model, best_params, mean_acc, mean_err, std_acc = cross_validate_svm(X_train, y_train)

    print(f"\nBest Parameters: {best_params}")
    print(f"Cross-Validation Mean Accuracy: {mean_acc:.4f}")
    print(f"Cross-Validation Std Dev: {std_acc:.4f}")
    print(f"Cross-Validation Mean Error: {mean_err:.4f}")

    # Step 3: Train best model on full training set and evaluate
    print("\nEvaluating on test set...")
    best_model.fit(X_train, y_train)
    y_pred = best_model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)

    print(f"Test Accuracy: {acc:.4f}")
    print("Classification Report:\n", report)

    return {
        'model': best_model,
        'cv_mean_accuracy': mean_acc,
        'cv_std_accuracy': std_acc,
        'cv_mean_error': mean_err,
        'test_accuracy': acc,
        'classification_report': report,
        'best_params': best_params
    }

In [5]:
act_path = "/om/scratch/Fri/imgriff/binaural_unit_activations_for_SVM/word_task_v10_main_feature_gain_config/word_task_v10_main_feature_gain_config_model_activations_for_word_SVM_train.h5"
h5_file = h5py.File(act_path, 'r', swmr=True)

RuntimeError: Can't deserialize object header prefix (bad object header version number)